# ViGoEmotions — Kaggle 27 nhãn, neural 20 epoch

Notebook độc lập chạy **ba mô hình cho 27 nhãn**: TF-IDF + OvR Logistic Regression, BiGRU và PhoBERT. Hai neural đều chạy đủ **20 epoch**; TF-IDF giữ solver liblinear/max_iter=1000, không có epoch neural tương đương.

Cả notebook 27 và 28 nhãn dùng duy nhất `configs/kaggle_equal_20ep.json`, cùng chọn checkpoint bằng validation Macro-F1 trên 27 cảm xúc, ngưỡng 0,5, hòa chọn epoch sớm. Notebook này có workspace, manifest, kết quả và ZIP riêng; không phải chờ hoặc chạy notebook còn lại.

Giữ Dataset/notebook Private. Attach `vigoemotions_kaggle_equal_20ep.zip` cùng ba CSV `id,text,labels`; bật GPU + Internet. Chạy tuần tự và dừng khi lỗi. Không chấm test, không dùng số baseline cũ. Chưa chạy GPU thì chưa có kết quả mới. Xem `docs/kaggle_equal_epochs.md`.

## 1. Nạp code vào workspace mới
Tạo ZIP trên máy bằng `python scripts/package_kaggle_equal_epochs.py`. Attach đúng một Dataset code và một Dataset dữ liệu. Nếu workspace đã tồn tại, không giải nén đè; dùng cell nối workspace sau restart kernel.

In [1]:
from pathlib import Path
import os, shutil, zipfile

INPUT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/ViGoEmotions27Equal20')
if ROOT.exists():
    raise FileExistsError(f'{ROOT} đã tồn tại. Không giải nén đè lên kết quả; chạy tiếp cell cần thiết hoặc dùng session mới.')

# Kaggle có thể giữ ZIP hoặc tự giải nén ZIP khi tạo Dataset.
archives = list(INPUT.rglob('vigoemotions_kaggle_equal_20ep.zip'))
code_roots = [p.parent.parent for p in INPUT.rglob('scripts/kaggle_equal_epochs.py')]
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        for item in archive.infolist():
            if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Unsafe ZIP path')
        ROOT.mkdir(parents=True)
        archive.extractall(ROOT)
elif not archives and len(code_roots) == 1:
    shutil.copytree(code_roots[0], ROOT)
else:
    raise RuntimeError('Hãy attach đúng một Dataset code chứa vigoemotions_kaggle_equal_20ep.zip hoặc project đã giải nén.')
os.chdir(ROOT)
print('Project:', ROOT)

Project: /kaggle/working/ViGoEmotions27Equal20


In [2]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path('/kaggle/working/ViGoEmotions27Equal20')
if not (ROOT/'scripts/kaggle_equal_epochs.py').is_file():
    raise FileNotFoundError('Nạp project trước; sau restart kernel chỉ chạy cell nối workspace này.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'scripts'))

## 2. Dependencies và GPU/Java
Internet phục vụ cài dependency và tải pretrained PhoBERT/VnCoreNLP. Một GPU được dùng cho mỗi lượt, các model chạy tuần tự. Không đổi loại GPU/package giữa study đã init.

In [3]:
%pip install -q -r requirements-colab.txt
import shutil, subprocess
if shutil.which('java') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'default-jre-headless'], check=True)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import torch, platform
assert torch.cuda.is_available(), 'Bật GPU Kaggle trước khi train neural.'
print('Python:', platform.python_version(), '| GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java', '-version'], check=True)

Python: 3.13.16 | GPU: Tesla T4


openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


CompletedProcess(args=['java', '-version'], returncode=0)

## 3. Nạp CSV riêng tư
Schema `id,text,labels`; bỏ qua CSV chỉ chứa ID trong gói code. Không hiển thị bình luận nguồn.

In [5]:
from pathlib import Path
import csv, shutil

sources = {}
for name in ('train.csv', 'val.csv', 'test.csv'):
    candidates = []
    for path in Path('/kaggle/input').rglob(name):
        with path.open(encoding='utf-8-sig', newline='') as handle:
            header = next(csv.reader(handle), None)
        # ZIP code có splits/*.csv chỉ chứa ID, không phải dữ liệu nguồn.
        if path.parent.name == 'splits' and header == ['id']:
            continue
        if header != ['id', 'text', 'labels']:
            raise ValueError(f'{path}: schema phải là id,text,labels')
        candidates.append(path)
    if len(candidates) != 1:
        raise RuntimeError(f'Cần đúng một {name} dữ liệu (id,text,labels) trong Input; tìm thấy {len(candidates)}. Kiểm tra Dataset đã attach.')
    sources[name] = candidates[0]
(ROOT / 'data').mkdir(exist_ok=True)
for name, source in sources.items():
    shutil.copy2(source, ROOT / 'data' / name)
    print('Đã nạp:', name)

DATA_DIR = ROOT/'data'

Đã nạp: train.csv
Đã nạp: val.csv
Đã nạp: test.csv


## 4. Init và đóng băng nguồn cấu hình
`OUT` là namespace mới riêng tư. Có thể đổi tên study con trước init; không đổi hyperparameters giữa study. Config/data/source/runtime khác sẽ dừng thay vì reuse nhầm kết quả cũ.

In [6]:
import json
import pandas as pd
from IPython.display import display

LABEL_COUNT = 27
OUT = ROOT/'reports/neutral_28_labels/kaggle_equal_20ep/labels27_study'
def run(action, *arguments):
    subprocess.run([sys.executable, '-u', str(ROOT/'scripts/kaggle_equal_epochs.py'), action,
                    '--output', str(OUT), '--data-dir', str(DATA_DIR),
                    '--labels', str(LABEL_COUNT), *map(str, arguments)],
                   cwd=ROOT, check=True)

run('init')
frozen = json.loads((OUT/'config27.json').read_text())
assert frozen['gru']['epochs'] == frozen['phobert']['epochs'] == 20
display(pd.DataFrame([dict(model=name, epochs=frozen[name]['epochs'],
                          batch_size=frozen[name]['batch_size']) for name in ('gru','phobert')]))
print('TF-IDF/LR:', frozen['logistic_regression'])
print('Checkpoint selection:', frozen['checkpoint_selection'])
print('Manifest:', json.loads((OUT/'manifest.json').read_text()))

{
  "status": "prepared_or_partial",
  "label_spaces": [
    27
  ],
  "completed_runs": 0,
  "expected_runs": 3,
  "neural_epochs": 20,
  "tfidf_budget": "Same OvR Logistic Regression solver/max_iter in both label spaces; epochs not applicable",
  "checkpoint_selection": "Highest validation emotion27 Macro-F1 at 0.5; ties keep earliest epoch",
  "source_of_truth": "This Kaggle manifest, config snapshots, run metadata and verified probabilities; no legacy scores imported",
  "limitation": "Equal epochs do not imply equal compute; GRU/PhoBERT use their documented batch sizes and architectures"
}


,model,epochs,batch_size
0,gru,20,64
1,phobert,20,16


TF-IDF/LR: {'C': 1.0, 'solver': 'liblinear', 'max_iter': 1000, 'random_state': 42}
Checkpoint selection: Highest validation emotion27 Macro-F1 at 0.5; ties keep earliest epoch
Manifest: {'profile': {'seed': 42, 'label_count': 27, 'neutral_id': 27, 'threshold': 0.5, 'split_policy': 'Preserve supplied val/test; remove normalized train overlaps; audit label prevalence', 'frequency_tiers': {'method': 'descending training frequency rank; tie by label ID', 'head_ranks': [1, 9], 'mid_ranks': [10, 18], 'tail_ranks': [19, 27]}, 'tfidf': {'max_features': 40000, 'ngram_range': [1, 2], 'min_df': 2, 'sublinear_tf': True, 'lowercase': True, 'token_pattern': '(?u)\\b\\w+\\b'}, 'logistic_regression': {'C': 1.0, 'solver': 'liblinear', 'max_iter': 1000, 'random_state': 42}, 'gru': {'max_vocab': 30000, 'embedding_dim': 128, 'hidden_dim': 128, 'dropout': 0.3, 'max_length': 128, 'epochs': 20, 'batch_size': 64, 'learning_rate': 0.001}, 'phobert': {'checkpoint': 'vinai/phobert-base-v2', 'max_length': 256, 'e

## 5. Kiểm thử offline giao thức
Fixture kiểm tra profile, tiêu chí chọn checkpoint, chống lẫn nguồn và output TF-IDF riêng. Không phải kết quả mô hình trên dataset thật.

In [7]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
                '-p', 'test_kaggle_equal_epochs.py', '-v'], cwd=ROOT, check=True)

test_canonical_profile_has_equal_neural_budget_and_shared_selection (test_kaggle_equal_epochs.EqualEpochTests.test_canonical_profile_has_equal_neural_budget_and_shared_selection) ... ok
test_checkpoint_criterion_ignores_neutral_column_only_in_paired_profile (test_kaggle_equal_epochs.EqualEpochTests.test_checkpoint_criterion_ignores_neutral_column_only_in_paired_profile) ... ok
test_each_label_scope_completes_independently_after_three_models (test_kaggle_equal_epochs.EqualEpochTests.test_each_label_scope_completes_independently_after_three_models) ... ok
test_empty_or_invalid_label_scope_is_not_a_completed_study (test_kaggle_equal_epochs.EqualEpochTests.test_empty_or_invalid_label_scope_is_not_a_completed_study) ... ok
test_kaggle_bootstrap_and_csv_discovery_work_offline (test_kaggle_equal_epochs.EqualEpochTests.test_kaggle_bootstrap_and_csv_discovery_work_offline) ... ok
test_missing_run_results_are_not_filled_from_legacy_reports (test_kaggle_equal_epochs.EqualEpochTests.test_missing_r

   model  label_count split  threshold  epochs_trained  best_epoch  macro_f1  micro_f1  hamming_loss  lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap  parameter_count  train_seconds            device                              source
tfidf_lr           27   val        0.5             NaN           3  0.037037       1.0           0.0   1.0            0.037037                 1.0                     0.0             1.0                1              1 synthetic fixture labels27/tfidf_lr/tfidf_lr_run.json
     gru           27   val        0.5            20.0           3  0.037037       1.0           0.0   1.0            0.037037                 1.0                     0.0             1.0                1              1 synthetic fixture           labels27/gru/gru_run.json
 phobert           27   val        0.5            20.0           3  0.037037       1.0           0.0   1.0            0.037037                 1.0                     0.0           

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_kaggle_equal_epochs.py', '-v'], returncode=0)

## Train 27 nhãn — TF-IDF + OvR Logistic Regression
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [8]:
run('train', '--model', 'tfidf_lr')

   model split  threshold  macro_f1  micro_f1  hamming_loss     lrap  parameter_count  train_seconds
tfidf_lr   val        0.5  0.238825  0.271238      0.060593 0.640745           940626        2.75236
   model  label_count split  threshold epochs_trained best_epoch  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap  parameter_count  train_seconds device                              source
tfidf_lr           27   val        0.5           None       None  0.238825  0.271238      0.060593 0.640745            0.238825            0.271238                0.060593        0.640745           940626        2.75236    cpu labels27/tfidf_lr/tfidf_lr_run.json
{
  "status": "prepared_or_partial",
  "label_spaces": [
    27
  ],
  "completed_runs": 1,
  "expected_runs": 3,
  "neural_epochs": 20,
  "tfidf_budget": "Same OvR Logistic Regression solver/max_iter in both label spaces; epochs not applicable",
  "checkpoint_selection":

## Train 27 nhãn — BiGRU — 20 epoch
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [9]:
run('train', '--model', 'gru')

Device: cuda
{'epoch': 1, 'loss': 0.27273018645180114, 'macro_f1': 0.03555485700722091, 'micro_f1': 0.05021340697966357, 'hamming_loss': 0.06781757556200925, 'lrap': 0.43555806570508115}
{'epoch': 2, 'loss': 0.2065822219206792, 'macro_f1': 0.17834929375776645, 'micro_f1': 0.2425156149041568, 'hamming_loss': 0.06304901222616614, 'lrap': 0.5648334368173931}
{'epoch': 3, 'loss': 0.17905382400956646, 'macro_f1': 0.27129847835997006, 'micro_f1': 0.34738682727802606, 'hamming_loss': 0.060216557312394677, 'lrap': 0.6024996917573717}
{'epoch': 4, 'loss': 0.16171961634071105, 'macro_f1': 0.3152110421763484, 'micro_f1': 0.391425908667288, 'hamming_loss': 0.058531425907998993, 'lrap': 0.619651950683619}
{'epoch': 5, 'loss': 0.14778557262021128, 'macro_f1': 0.3439695478465623, 'micro_f1': 0.4115942028985507, 'hamming_loss': 0.058226668100821055, 'lrap': 0.6299632642860445}
{'epoch': 6, 'loss': 0.13532600640277032, 'macro_f1': 0.3581291253857337, 'micro_f1': 0.41612670986321093, 'hamming_loss': 0.0

## Train 27 nhãn — PhoBERT — 20 epoch
Model hoàn tất được kiểm chứng và giữ nguyên khi chạy lại cell; model dở dang không resume giữa epoch, dùng study mới để chạy lại.

In [10]:
run('train', '--model', 'phobert')

--2026-11-05 04:41:01--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 3.97M 7s
    50K .......... .......... .......... .......... ..........  0% 4.81M 6s
   100K .......... .......... .......... .......... ..........  0% 20.6M 4s
   150K .......... .......... .......... .......... ..........  0% 21.5M 4s
   200K .......... .......... .......... .......... ..........  0% 26.2M 3s
   250K .......... .......... .......... .......... ..........  1% 9.48M 3s
   300K .......... .......... .......... .......... ..........  1% 40.8M 3s
   350K ..

Device: cuda
2026-11-05 04:41:15 INFO  WordSegmenter:24 - Loading Word Segmentation model
I0000 00:00:1793853709.968271     210 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
{'epoch': 1, 'loss': 0.26424857017421316, 'macro_f1': 0.0, 'micro_f1': 0.0, 'hamming_loss': 0.06926965687856297, 'lrap': 0.3993900523623596}
{'epoch': 2, 'loss': 0.21100333204286717, 'macro_f1': 0.08002160250662896, 'micro_f1': 0.24405377456049637, 'hamming_loss': 0.06552292854325768, 'lrap': 0.581104339589491}
{'epoch': 3, 'loss': 0.17012267322231622, 'macro_f1': 0.3342832260499033, 'micro_f1': 0.458412918909532

## 6. Kết quả ba mô hình 27 nhãn
Bảng chỉ dùng TF-IDF/BiGRU/PhoBERT của notebook này, không cần kết quả notebook còn lại để hoàn tất hoặc tải ZIP. Khi đối chiếu 27/28 sau này, dùng emotion27_* và kiểm tra profile, source code, CSV hash, fixed IDs; công bố khác biệt GPU/package nếu hai session khác nhau. Không nhập điểm lịch sử vào bảng.

In [11]:
run('report')
status = json.loads((OUT/'status.json').read_text())
assert status['status'] == 'completed' and status['expected_runs'] == 3, 'Hoàn tất đủ ba model 27 nhãn trước khi archive.'
display(pd.read_csv(OUT/'comparison.csv'))
print(status)

   model  label_count split  threshold  epochs_trained  best_epoch  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap  parameter_count  train_seconds   device                              source
tfidf_lr           27   val        0.5             NaN         NaN  0.238825  0.271238      0.060593 0.640745            0.238825            0.271238                0.060593        0.640745           940626       2.752360      cpu labels27/tfidf_lr/tfidf_lr_run.json
     gru           27   val        0.5            20.0        17.0  0.421976  0.459037      0.062619 0.619133            0.421976            0.459037                0.062619        0.619133          2128283      80.330702 Tesla T4           labels27/gru/gru_run.json
 phobert           27   val        0.5            20.0        14.0  0.597410  0.599582      0.055000 0.741945            0.597410            0.599582                0.055000        0.741945        13

,model,label_count,split,threshold,epochs_trained,best_epoch,macro_f1,micro_f1,hamming_loss,lrap,emotion27_macro_f1,emotion27_micro_f1,emotion27_hamming_loss,emotion27_lrap,parameter_count,train_seconds,device,source
0,tfidf_lr,27,val,0.5,NaN,NaN,0.238825,0.271238,0.060593,0.640745,0.238825,0.271238,0.060593,0.640745,940626,2.752360,cpu,labels27/tfidf_lr/tfidf_lr_run.json
1,gru,27,val,0.5,20.0,17.0,0.421976,0.459037,0.062619,0.619133,0.421976,0.459037,0.062619,0.619133,2128283,80.330702,Tesla T4,labels27/gru/gru_run.json
2,phobert,27,val,0.5,20.0,14.0,0.597410,0.599582,0.055000,0.741945,0.597410,0.599582,0.055000,0.741945,135019035,14589.618177,Tesla T4,labels27/phobert/phobert_run.json


{'status': 'completed', 'label_spaces': [27], 'completed_runs': 3, 'expected_runs': 3, 'neural_epochs': 20, 'tfidf_budget': 'Same OvR Logistic Regression solver/max_iter in both label spaces; epochs not applicable', 'checkpoint_selection': 'Highest validation emotion27 Macro-F1 at 0.5; ties keep earliest epoch', 'source_of_truth': 'This Kaggle manifest, config snapshots, run metadata and verified probabilities; no legacy scores imported', 'limitation': 'Equal epochs do not imply equal compute; GRU/PhoBERT use their documented batch sizes and architectures'}


## 7. Tải toàn bộ bằng chứng/checkpoint riêng tư
ZIP gồm manifest, config, log neural, history, probabilities, checkpoint và vocabulary; không kèm CSV/bình luận nguồn. Giữ Private vì có vocabulary/checkpoint. Tải trước khi mất runtime; không dùng ZIP này làm gói công khai. Không có cell chấm test.

Sau khi tải về, dùng ZIP này để nhập/kiểm chứng và cập nhật report; không sửa tay số điểm cũ thành điểm mới.

In [12]:
run('archive')
archive = ROOT/'vigoemotions_kaggle_equal20_27labels_private_results.zip'
assert archive.is_file()
print('Tải từ Kaggle Output:', archive, '| MiB:', round(archive.stat().st_size/1024**2, 2))

   model  label_count split  threshold  epochs_trained  best_epoch  macro_f1  micro_f1  hamming_loss     lrap  emotion27_macro_f1  emotion27_micro_f1  emotion27_hamming_loss  emotion27_lrap  parameter_count  train_seconds   device                              source
tfidf_lr           27   val        0.5             NaN         NaN  0.238825  0.271238      0.060593 0.640745            0.238825            0.271238                0.060593        0.640745           940626       2.752360      cpu labels27/tfidf_lr/tfidf_lr_run.json
     gru           27   val        0.5            20.0        17.0  0.421976  0.459037      0.062619 0.619133            0.421976            0.459037                0.062619        0.619133          2128283      80.330702 Tesla T4           labels27/gru/gru_run.json
 phobert           27   val        0.5            20.0        14.0  0.597410  0.599582      0.055000 0.741945            0.597410            0.599582                0.055000        0.741945        13